In [1]:
print("hello")

hello


In [1]:
# SparkSession - bramka do Sparka
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
df.describe().show()

+-------+-----------------+-----------+-------+-------+-------+
|summary|           amount|   category|  store|  tx_id|user_id|
+-------+-----------------+-----------+-------+-------+-------+
|  count|            10000|      10000|  10000|  10000|  10000|
|   mean|401.1544750000007|       NULL|   NULL|   NULL|   NULL|
| stddev|635.5781805674461|       NULL|   NULL|   NULL|   NULL|
|    min|              5.0|elektronika| Gdańsk|TX00001|    u01|
|    max|           9999.0|    żywność|Wrocław|TX10000|    u50|
+-------+-----------------+-----------+-------+-------+-------+



In [6]:
df.show(10)

+------+-----------+--------+-------------------+-------+-------+
|amount|   category|   store|          timestamp|  tx_id|user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|    u48|
| 79.57|    książki|Warszawa|2026-04-12 08:05:43|TX00002|    u15|
|126.17|     odzież|Warszawa|2026-04-12 09:15:30|TX00003|    u18|
| 34.08|     odzież|Warszawa|2026-04-12 10:05:39|TX00004|    u10|
|428.88|    żywność|  Kraków|2026-04-12 09:04:36|TX00005|    u17|
|345.21|    książki|Warszawa|2026-04-12 09:36:31|TX00006|    u25|
|376.42|    żywność|Warszawa|2026-04-12 10:06:49|TX00007|    u15|
| 85.36|elektronika|  Gdańsk|2026-04-12 09:08:25|TX00008|    u24|
| 66.26|    żywność|  Kraków|2026-04-12 10:06:19|TX00009|    u05|
|660.41|     odzież|  Kraków|2026-04-12 08:29:24|TX00010|    u41|
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [7]:
df.toPandas()

,amount,category,store,timestamp,tx_id,user_id
0,312.32,elektronika,Warszawa,2026-04-12 08:25:07,TX00001,u48
1,79.57,książki,Warszawa,2026-04-12 08:05:43,TX00002,u15
2,126.17,odzież,Warszawa,2026-04-12 09:15:30,TX00003,u18
3,34.08,odzież,Warszawa,2026-04-12 10:05:39,TX00004,u10
4,428.88,żywność,Kraków,2026-04-12 09:04:36,TX00005,u17
...,...,...,...,...,...,...
9995,98.65,żywność,Gdańsk,2026-04-12 09:19:43,TX09996,u41
9996,221.05,książki,Wrocław,2026-04-12 08:43:07,TX09997,u17
9997,78.88,elektronika,Wrocław,2026-04-12 08:35:51,TX09998,u01
9998,212.63,elektronika,Gdańsk,2026-04-12 09:28:45,TX09999,u02


In [8]:
df.toPandas().describe()

,amount,timestamp
count,10000.000000,10000
mean,401.154475,2026-04-12 09:24:21.459800320
min,5.000000,2026-04-12 08:00:06
25%,98.880000,2026-04-12 08:52:04
50%,210.080000,2026-04-12 09:22:57.500000
75%,449.745000,2026-04-12 09:55:22.249999872
max,9999.000000,2026-04-12 10:59:57
std,635.578181,NaN


In [9]:
# Zadanie 2.1 - Liczba transakcji i suma przychodów per sklep
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

In [10]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [11]:
# Zadanie 2.2 - Statystyki per kategoria
# Policz sumę, minimum i maksimum kwoty dla każdej kategorii
from pyspark.sql.functions import sum as _sum, min as _min, max as _max, round as _round

df.groupBy("category") \
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN")
    ) \
    .orderBy("category") \
    .show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [12]:
# Zadanie 3.1 - Liczba transakcji per godzina (tumbling 1h)
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))   
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [13]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [14]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [15]:
# Zadanie 3.2 - okna 30-minutowe per sklep
# Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. Posortuj po oknie, a w ramach okna po sklepie.
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

window_30min = (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)

window_30min.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("store"),
    col("liczba_tx"),
    col("suma_PLN")
).show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [16]:
# Zadanie 3.3 - W której godzinie sklep "Kraków' miał najwyższy przychód?
# Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejaco po sumie.
from pyspark.sql.functions import window, sum as _sum, round as _round, col, desc

krakow_suma = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy(desc("suma_PLN"))
)

krakow_suma.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("suma_PLN")
).show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|201259.26|
+-------------------+-------------------+---------+



In [17]:
# Zadanie 4.1 - Okno 1h, krok 30 minut
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [18]:
# Zadanie 4.2 - Porównaj tumbling vs sliding 
# Policz łaczna liczbę wierszy wynikowych w obu podejściach. Dlaczego sliding daje więcej wierszy?
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: 
# Sliding ma więcej wierszy, ponieważ okna nachodza na siebie. 
# Nowe okno uruchamia się co 30 minut, ale każde trwa godzinę. Oznacza to, że ten sam rekord może należeć jednocześnie do dwóch okien.
# Z kolei przy Tumbling mamy rozłaczne okna 1-godzinne bez nakładania obserwacji, więc liczba wierszy jest mniejsza.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Część 5: Pytania kontrolne
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661 transakcji

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ:
# groupBy("store") grupuje dane tylko według sklepu,
# więc dostajemy jedną agregację dla każdego sklepu z całego zbioru danych.
# groupBy(window(...), "store") grupuje jednocześnie po czasie i sklepie,
# zatem dla każdego sklepu powstają osobne wyniki w kolejnych oknach czasowych
# (np. co godzinę lub co 30 minut).

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ:
# Transakcje z godziny 09:30 należą do 2 okien sliding.
# Przy oknie 1h przesuwanym co 30 min są to:
# 09:00–10:00
# 09:30–10:30
# Dzieje się tak, ponieważ okna nachodzą na siebie co 30 minut.

In [19]:
# PRACA DOMOWA
# Zadanie 1.
# Znajdź godzinę, w której sklep Gdańsk miał najniższa średnia kwotę transakcji

from pyspark.sql.functions import window, avg, round as _round, col, asc

gdansk_min = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .orderBy(asc("srednia_PLN"))
    .limit(1)
)

gdansk_min.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("srednia_PLN")
).show(truncate=False)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
+-------------------+-------------------+-----------+



In [20]:
# Zadanie 2.
# Policz ile transakcji per kategoria było w oknie 9:00-9:30

from pyspark.sql.functions import count, hour, minute, col

df.filter(
    (hour(col("timestamp")) == 9) &
    (minute(col("timestamp")) < 30)
).groupBy("category") \
 .agg(count("tx_id").alias("liczba_tx")) \
 .orderBy("category") \
 .show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [21]:
# Zadanie 3.
# Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji.

from pyspark.sql.functions import window, count, col, desc

szczyt_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
)

szczyt_15min.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("liczba_tx")
).orderBy(desc("liczba_tx")).show(truncate=False)

print("Szczyt transakcji:")
szczyt = szczyt_15min.orderBy(desc("liczba_tx")).limit(1)

szczyt.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    col("liczba_tx")
).show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+

Szczyt transakcji:
+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---